In [ ]:
import cracknuts as cn

In [ ]:
# Create a cracker instance
s1 = cn.cracker_s1('192.168.0.15')

In [ ]:
# Connect to the cracker device
s1.connect(force_update_bin=True, force_write_default_config=True)

In [ ]:
import random
import time
from cracknuts.cracker import serial


cmd_set_aes_enc_key = "01 00 00 00 00 00 00 10"
cmd_aes_enc = "01 02 00 00 00 00 00 10"

aes_key = "11 22 33 44 55 66 77 88 99 00 aa bb cc dd ee ff"
aes_data_len = 16

sample_length = 4000

def init(cracker):
    cracker.nut_voltage_enable()
    cracker.nut_voltage(3.3)
    cracker.nut_clock_enable()
    cracker.nut_clock_freq('8M')
    cracker.uart_io_enable()
    cracker.osc_sample_clock('48m')
    cracker.osc_sample_length(sample_length)
    cracker.osc_trigger_source('N')
    cracker.osc_analog_gain('B', 10)
    cracker.osc_trigger_level(0)
    cracker.osc_trigger_mode('E')
    cracker.osc_trigger_edge('U')
    cracker.uart_config(baudrate=serial.Baudrate.BAUDRATE_115200, bytesize=serial.Bytesize.EIGHTBITS, parity=serial.Parity.PARITY_NONE, stopbits=serial.Stopbits.STOPBITS_ONE)

    time.sleep(2)
    cmd = cmd_set_aes_enc_key + aes_key
    cracker.uart_transmit_receive(cmd, timeout=1000, rx_count=6)

def do(cracker, count):
    plaintext_data = random.randbytes(aes_data_len)
    tx_data = bytes.fromhex(cmd_aes_enc.replace(' ', '')) + plaintext_data
    status, ret = cracker.uart_transmit_receive(tx_data, rx_count= 6 + aes_data_len, is_trigger=True)
    
    return {
        "plaintext": plaintext_data,
        "ciphertext": ret[-aes_data_len:],
        "key": bytes.fromhex(aes_key)
    }


def finish(c):
    ...
    # print('optional behavior')


acq = cn.simple_acq(s1, init_func=init, do_func=do, finish_func=finish)

In [ ]:
cn.show_panel(acq)

In [ ]:
ds = cn.load_trace_dataset('./dataset/20260123151941.zarr/')

In [ ]:
ds.info()

In [ ]:
ds.plot()

In [1]:
import zarr
zd = zarr.open('./dataset/20260123151941.zarr/')
zd.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='0', nodes=(Node(disabled=True, name…

In [ ]:
!pip install ipytree